# Masked Diffusion Language Model - Training on Kaggle

Based on: **Discrete Diffusion Language Model for Efficient Text Summarization** (NAACL 2025)

**Requirements**: GPU runtime (T4 16GB recommended)

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone repository (branch with new architecture)
# Remove old clone if exists (important: get latest fixes!)
!rm -rf Final_qualification_work
!git clone -b feature/architecture_update https://github.com/KrugD/Final_qualification_work.git
%cd Final_qualification_work
!ls

In [ ]:
# Install dependencies
!pip install accelerate transformers datasets torch pyyaml python-dotenv tqdm numpy sentencepiece protobuf rouge-score bert-score comet_ml pandas --quiet

# Note: mamba-ssm is incompatible with PyTorch 2.8 on Kaggle.
# FallbackMamba (gated convolution) is used automatically — same training quality.

In [ ]:
# Setup CometML API key
import os

# Option 1: Set directly (replace with your key)
os.environ["COMET_API_KEY"] = "YOUR_COMET_API_KEY"  # <-- replace this
os.environ["COMET_PROJECT_NAME"] = "diffusion-summarization"

# Option 2: From Kaggle Secrets (recommended)
# from kaggle_secrets import UserSecretsClient
# secrets = UserSecretsClient()
# os.environ["COMET_API_KEY"] = secrets.get_secret("COMET_API_KEY")

# Write .env file
with open(".env", "w") as f:
    f.write(f'COMET_API_KEY={os.environ["COMET_API_KEY"]}\n')
    f.write(f'COMET_PROJECT_NAME={os.environ["COMET_PROJECT_NAME"]}\n')

print("Environment configured!")

In [ ]:
# Verify installation
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

try:
    from mamba_ssm import Mamba
    print("mamba-ssm: installed")
except ImportError:
    print("mamba-ssm: NOT installed (will use fallback)")

from src.model import MaskedDiffusionSummarizer
print("\nProject modules loaded successfully!")

## 2. Train from Scratch

In [ ]:
# Start training
!python train.py --config config/kaggle_config.yaml

## 2b. Resume Training (if session was interrupted)

In [ ]:
# List available checkpoints
import os
from pathlib import Path

checkpoint_dir = Path("/kaggle/working/checkpoints")
if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.glob("checkpoint_*"))
    print("Available checkpoints:")
    for cp in checkpoints:
        print(f"  {cp}")
    
    if checkpoints:
        latest = checkpoints[-1]
        print(f"\nLatest checkpoint: {latest}")
        print(f"\nTo resume, run:")
        print(f"  !python train.py --config config/kaggle_config.yaml --resume {latest}")
else:
    print("No checkpoints found. Start training from scratch (Cell 2).")

In [ ]:
# Resume from the latest checkpoint (uncomment and update path)
# !python train.py --config config/kaggle_config.yaml --resume /kaggle/working/checkpoints/checkpoint_epoch1_step2000

## 3. Check Results

In [ ]:
import torch
from pathlib import Path

checkpoint_dir = Path("/kaggle/working/checkpoints")

# Load best model metrics
best_metrics_path = checkpoint_dir / "best_model" / "best_metrics.pt"
if best_metrics_path.exists():
    metrics = torch.load(best_metrics_path, weights_only=False)
    print("=" * 50)
    print("BEST MODEL METRICS")
    print("=" * 50)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
else:
    print("Best model not yet saved (training in progress).")

# Load training summary
summary_path = checkpoint_dir / "training_summary.pt"
if summary_path.exists():
    summary = torch.load(summary_path, weights_only=False)
    print("\n" + "=" * 50)
    print("TRAINING SUMMARY")
    print("=" * 50)
    for k, v in summary.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

## 4. Test Generation with Best Model

In [ ]:
import torch
from src.model import MaskedDiffusionSummarizer
from transformers import AutoTokenizer

# Load best model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MaskedDiffusionSummarizer.from_pretrained(
    "/kaggle/working/checkpoints/best_model/weights",
    device=str(device),
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained("ai-forever/ruT5-base")
print(f"Model loaded on {device}")

In [ ]:
# Test on a sample text
test_text = """Российские учёные из Института ядерной физики СО РАН разработали 
новый метод диагностики материалов с помощью синхротронного излучения. 
Метод позволяет исследовать внутреннюю структуру объектов без их разрушения. 
Технология может применяться в медицине, промышленности и археологии. 
Результаты исследования опубликованы в журнале Nature Materials."""

# Tokenize
inputs = tokenizer(
    test_text,
    max_length=512,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
).to(device)

# Generate summary
with torch.no_grad():
    generated_ids, confidence = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128,
        num_inference_steps=10,
    )

summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("Source:")
print(test_text)
print("\nGenerated summary:")
print(summary)
print(f"\nConfidence: {confidence[0].mean():.4f}")

## 5. Download Best Weights

Download the best model weights to use locally.

In [ ]:
import shutil
from pathlib import Path

# Package best weights for download
best_weights_dir = Path("/kaggle/working/checkpoints/best_model/weights")
output_archive = Path("/kaggle/working/best_model_weights")

if best_weights_dir.exists():
    # Show what will be packaged
    print("Files in best model weights:")
    total_size = 0
    for f in sorted(best_weights_dir.rglob("*")):
        if f.is_file():
            size_mb = f.stat().st_size / 1024 / 1024
            total_size += size_mb
            print(f"  {f.name}: {size_mb:.1f} MB")
    print(f"\nTotal: {total_size:.1f} MB")
    
    # Also copy best metrics
    metrics_file = Path("/kaggle/working/checkpoints/best_model/best_metrics.pt")
    if metrics_file.exists():
        shutil.copy2(metrics_file, best_weights_dir / "best_metrics.pt")
    
    # Create archive
    shutil.make_archive(str(output_archive), "zip", best_weights_dir)
    archive_size = Path(f"{output_archive}.zip").stat().st_size / 1024 / 1024
    print(f"\nArchive created: {output_archive}.zip ({archive_size:.1f} MB)")
    print("\nDownload from Kaggle Output tab or run:")
    print("  from IPython.display import FileLink")
    print("  FileLink('/kaggle/working/best_model_weights.zip')")
else:
    print("Best model weights not found. Complete training first.")

In [ ]:
# Display download link
from IPython.display import FileLink
FileLink("/kaggle/working/best_model_weights.zip")